# 09 — ML: Series de Tiempo (SARIMA) — Dashboard 7

**Objetivo:** Predecir la demanda horaria de viajes para los próximos 7 días.

**Flujo:**
1. Lee `mart_demand_volume` (ya agregado, liviano) desde `tlc_gold`.
2. Construye la serie de tiempo horaria por tipo de vehículo.
3. Aplica test ADF para confirmar no-estacionariedad.
4. Analiza ACF/PACF para detectar estacionalidades (s=24h, s=168h).
5. Ajusta SARIMA(1,1,1)(1,1,1)[24].
6. Genera pronóstico de 7 días con intervalos de confianza al 95%.
7. Guarda predicciones en `tlc_gold.ml_sarima_forecast`.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from core.config.settings import settings
import pyspark.sql.functions as F

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
import joblib
from pathlib import Path

MODELS_DIR = Path('/home/jovyan/work/data/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR = Path('/home/jovyan/work/data/charts')
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK.')

In [ ]:
spark = get_spark('TLC_ML_SARIMA')
print(f'Spark {spark.version} ready.')

In [ ]:
# Leer mart ya agregado — pocos miles de filas, no millones
mart_df = read_mongo(spark, settings.MONGO_DB_GOLD, 'mart_demand_volume')

# Construir serie de tiempo horaria para TODOS los vehículos
hourly_spark = (
    mart_df
    .groupBy('year', 'month', 'day', 'hour')
    .agg(F.sum('total_trips').alias('demand'))
    .withColumn(
        'pickup_dt',
        F.to_timestamp(
            F.concat_ws('-',
                F.col('year').cast('string'),
                F.lpad(F.col('month').cast('string'), 2, '0'),
                F.lpad(F.col('day').cast('string'), 2, '0'),
                F.lpad(F.col('hour').cast('string'), 2, '0'),
            ),
            'yyyy-MM-dd-HH'
        )
    )
    .orderBy('pickup_dt')
    .select('pickup_dt', 'demand')
)

# Convertir a Pandas para statsmodels (Arrow vectorizado)
ts_pd = hourly_spark.toPandas()
ts_pd.set_index('pickup_dt', inplace=True)
ts_pd.index = pd.to_datetime(ts_pd.index)
ts_pd = ts_pd.sort_index().asfreq('h', fill_value=0)

print(f'Serie de tiempo: {len(ts_pd):,} observaciones horarias')
print(f'Rango: {ts_pd.index.min()} → {ts_pd.index.max()}')
print(ts_pd['demand'].describe().round(1))

In [ ]:
# ── Gráfica 1: Serie histórica completa
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(ts_pd.index, ts_pd['demand'], linewidth=0.5, alpha=0.8, color='#3b82f6')
ax.set_title('Demanda Horaria Total — NYC TLC (2023–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Viajes / Hora')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'sarima_01_raw_series.png'), dpi=120)
plt.close()
print('Gráfica 1 guardada: sarima_01_raw_series.png')

In [ ]:
# ── Test ADF (Augmented Dickey-Fuller)
adf_result  = adfuller(ts_pd['demand'].dropna(), autolag='AIC')
adf_stat    = adf_result[0]
p_value     = adf_result[1]
crit_vals   = adf_result[4]

print('═' * 45)
print('  Augmented Dickey-Fuller Test')
print('═' * 45)
print(f'  ADF Statistic : {adf_stat:.4f}')
print(f'  p-value       : {p_value:.6f}')
for k, v in crit_vals.items():
    print(f'  Critical ({k}) : {v:.4f}')

if p_value > 0.05:
    print('\n✗ RESULTADO: No estacionaria → Diferenciación necesaria → SARIMA')
else:
    print('\n✓ RESULTADO: Estacionaria (d=0) → SARIMA aún recomendado por estacionalidad')

In [ ]:
# ── ACF / PACF
ts_diff = ts_pd['demand'].diff().dropna()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
plot_acf(ts_diff, lags=200, ax=axes[0], color='#3b82f6',
         title='ACF — Demanda Diferenciada (lags=200)')
axes[0].axvline(x=24,  color='red',    linestyle='--', alpha=0.7, label='lag=24h')
axes[0].axvline(x=168, color='orange', linestyle='--', alpha=0.7, label='lag=168h (7 días)')
axes[0].legend()

plot_pacf(ts_diff, lags=50, ax=axes[1], method='ywm', color='#10b981',
          title='PACF — Demanda Diferenciada (lags=50)')
axes[1].axvline(x=24, color='red', linestyle='--', alpha=0.7, label='lag=24h')
axes[1].legend()

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'sarima_02_acf_pacf.png'), dpi=120)
plt.close()
print('Gráfica 2 guardada: sarima_02_acf_pacf.png')

In [ ]:
# ── Entrenar SARIMA (30 días de datos para velocidad de demo)
TRAIN_DAYS  = 30
train_series = ts_pd['demand'].iloc[-(TRAIN_DAYS * 24):]

sarima_order          = (1, 1, 1)
sarima_seasonal_order = (1, 1, 1, 24)  # s=24h: estacionalidad diaria

print(f'Entrenando SARIMA{sarima_order}×{sarima_seasonal_order}...')
print(f'  Período: {train_series.index[0]} → {train_series.index[-1]}')
print(f'  Observaciones: {len(train_series):,}')

model = SARIMAX(
    train_series,
    order=sarima_order,
    seasonal_order=sarima_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = model.fit(disp=False)

print(f'\n  AIC : {sarima_fit.aic:.2f}')
print(f'  BIC : {sarima_fit.bic:.2f}')

# Persistir modelo
model_path = MODELS_DIR / 'sarima_demand_model.pkl'
joblib.dump(sarima_fit, model_path, compress=3)
print(f'\n✓ Modelo guardado en: {model_path}')

In [ ]:
# ── Pronóstico 7 días
FORECAST_HOURS = 7 * 24
forecast       = sarima_fit.get_forecast(steps=FORECAST_HOURS)
forecast_mean  = forecast.predicted_mean
conf_int       = forecast.conf_int(alpha=0.05)

print(f'Pronóstico generado: {FORECAST_HOURS}h ({FORECAST_HOURS // 24} días)')
print(f'Demanda promedio predicha: {forecast_mean.mean():.0f} viajes/hora')

# Gráfica 3: Forecast
fig, ax = plt.subplots(figsize=(16, 5))
history = train_series.iloc[-(7 * 24):]
ax.plot(history.index, history.values, label='Histórico (últimos 7 días)', color='#3b82f6', linewidth=1.2)
ax.plot(forecast_mean.index, forecast_mean.values, label='Pronóstico SARIMA (7 días)', color='#f59e0b', linewidth=1.5, linestyle='--')
ax.fill_between(forecast_mean.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                color='#f59e0b', alpha=0.15, label='IC 95%')
ax.axvline(x=forecast_mean.index[0], color='red', linestyle=':', alpha=0.7, label='Inicio Pronóstico')
ax.set_title(f'SARIMA{sarima_order}×{sarima_seasonal_order} — Pronóstico Demanda NYC TLC (7 días)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Viajes / Hora')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'sarima_03_forecast.png'), dpi=150)
plt.close()
print('Gráfica 3 guardada: sarima_03_forecast.png')

In [ ]:
# ── Guardar pronóstico en MongoDB Gold
forecast_pd = pd.DataFrame({
    'forecast_dt':        forecast_mean.index,
    'predicted_demand':   forecast_mean.values.round(0).astype(int),
    'ci_lower':           conf_int.iloc[:, 0].values.round(0).astype(int),
    'ci_upper':           conf_int.iloc[:, 1].values.round(0).astype(int),
    'model':              f'SARIMA{sarima_order}x{sarima_seasonal_order}',
    'train_days':         TRAIN_DAYS,
    'generated_at':       pd.Timestamp.now(),
})

forecast_spark = spark.createDataFrame(forecast_pd)
write_mongo(forecast_spark, settings.MONGO_DB_GOLD, 'ml_sarima_forecast', mode='overwrite')
print(f'✓ {len(forecast_pd)} filas guardadas en tlc_gold.ml_sarima_forecast')